# Acute inhibitor response analysis of transcription and locus-centered SNAP signals

This notebook analyzes HDAC-inhibitor and THZ1 experiments as separate comparison branches. It reads Cell 4 measurements, corrects SNAP photobleaching with matched DMSO cells, summarizes quality control, and draws one main and one supplementary figure for each replicate.

The primary figure combines transcriptional Active fraction with background-subtracted local SNAP. Nuclear SNAP is shown in the supplementary figure. Raw signals and correction factors are saved for inspection.


## Data sources and output folder

The experiment table defines the date, internal replicate identifier, target gene, SNAP label, and treatment comparison for every dataset. Cell 4 tables are read exclusively from the isolated physical-distance reanalysis output. The combined 260727 datasets are split into separate HDACi and THZ1 comparisons.

By default, new tables and figures are written to a `publication_outputs/final` folder beside the notebook folder. The input and output roots can be changed with the environment variables shown below.

In [ ]:
from pathlib import Path
import os

# Full reprocessing requires the upstream Cell 4/QC tables generated from the
# controlled microscopy data. These are selected explicitly rather than through
# a machine-specific default. The Zenodo revision archive contains the final
# per-cell and summary tables used for Supplementary Fig. 13.
RAW_ROOT = Path(os.environ.get("GAO2026_RAW_ROOT", "external_raw")).expanduser().resolve()
DATA_ROOT = Path(os.environ.get(
    "LIVE_INHIBITOR_DATA_ROOT",
    str(RAW_ROOT / "live_inhibitor"),
)).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("GAO2026_OUTPUT_ROOT", "outputs")).expanduser().resolve()
LOCAL_ROOT = Path(os.environ.get(
    "LIVE_INHIBITOR_OUTPUT_ROOT",
    str(OUTPUT_ROOT / "14_acute_inhibitor"),
)).expanduser().resolve()

EXPERIMENTS = [
    dict(key="260717_Nanog_H3K27ac", rel="260717-live-inhibitor/Nanog", date="260717", replicate=1, gene="Nanog", snap="H3K27ac", design="HDACi"),
    dict(key="260723_Sox2_H3K27ac", rel="260723-live-inhibitor/Sox2", date="260723", replicate=1, gene="Sox2", snap="H3K27ac", design="HDACi"),
    dict(key="260727_Nanog_H3K27ac", rel="260727-live-inhibitor/Nanog", date="260727", replicate=2, thz1_replicate=1, gene="Nanog", snap="H3K27ac", design="HDACi+THZ1"),
    dict(key="260727_Sox2_H3K27ac", rel="260727-live-inhibitor/Sox2", date="260727", replicate=2, thz1_replicate=1, gene="Sox2", snap="H3K27ac", design="HDACi+THZ1"),
    dict(key="260729_Nanog_HDAC1", rel="260729-live-inhibitor/HDAC1", date="260729", replicate=1, gene="Nanog", snap="HDAC1", design="THZ1"),
    dict(key="260729_Nanog_HDAC3", rel="260729-live-inhibitor/HDAC3", date="260729", replicate=1, gene="Nanog", snap="HDAC3", design="THZ1"),
    dict(key="260729_Nanog_H3K27ac", rel="260729-live-inhibitor/Nanog-H3K27ac", date="260729", replicate=2, gene="Nanog", snap="H3K27ac", design="THZ1"),
    dict(key="260731_Nanog_p300", rel="260731-live-inhibitor/P300", date="260731", replicate=1, gene="Nanog", snap="P300", display_snap="p300", design="THZ1"),
    dict(key="260801_Nanog_HDAC1", rel="260801-live-inhibitor/HDAC1", date="260801", replicate=2, gene="Nanog", snap="HDAC1", design="THZ1"),
    dict(key="260801_Nanog_p300", rel="260801-live-inhibitor/P300", date="260801", replicate=2, gene="Nanog", snap="P300", display_snap="p300", design="THZ1"),
    dict(key="260805_Nanog_HDAC3", rel="260805-live-inhibitor/HDAC3", date="260805", replicate=2, gene="Nanog", snap="HDAC3", design="THZ1"),
    dict(key="260805_Sox2_H3K27ac", rel="260805-live-inhibitor/Sox2-H3K27ac", date="260805", replicate=2, gene="Sox2", snap="H3K27ac", design="THZ1"),
    dict(key="260806_Nanog_HDAC3", rel="260806-live-inhibitor/HDAC3", date="260806", replicate=3, gene="Nanog", snap="HDAC3", design="THZ1"),
    dict(key="260807_Nanog_HDAC1", rel="260807-live-inhibitor/HDAC1", date="260807", replicate=3, gene="Nanog", snap="HDAC1", design="THZ1"),
    dict(key="260807_Sox2_H3K27ac", rel="260807-live-inhibitor/Sox2-H3K27ac", date="260807", replicate=3, gene="Sox2", snap="H3K27ac", design="HDACi"),
]
EXPERIMENTS = {row["key"]: row for row in EXPERIMENTS}

VALID_FRAMES = (0, 6, 12, 18, 24)
TIME_MIN = (0.0, 15.0, 30.0, 45.0, 60.0)
FRAME_TO_TIME = dict(zip(VALID_FRAMES, TIME_MIN))
GROUP_MAP = {"well001": "DMSO", "well002": "TSA", "well003": "RGFP966", "well004": "THZ1"}
THZ1_GROUP_MAP = {"well001": "DMSO", "well002": "THZ1"}


## Plot settings and Cell 4 loading

Figures use Arial when available, 7 pt base text, outward ticks, no grid, and editable TrueType text in PDF files. Treatment colors are fixed for all replicates.

The loader selects the non-empty Cell 4 table for each experiment and adds the experiment metadata. Available CNR, low-expression, and saturation fields are merged from the existing quality-control table.


In [ ]:
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

RESULT_ROOT = LOCAL_ROOT / "final"
TABLE_ROOT = RESULT_ROOT / "tables"
FIGURE_ROOT = RESULT_ROOT / "figures"
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_ROOT.mkdir(parents=True, exist_ok=True)

COLORS = {"DMSO": "#377EB8", "TSA": "#FF7F00", "RGFP966": "#4DAF4A", "THZ1": "#984EA3"}
MARKERS = {1: "o", 2: "s", 3: "^"}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "legend.frameon": False,
    "axes.linewidth": 0.8,
    "axes.grid": False,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
})

def style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(length=3, width=0.8)

def find_cell4_table(exp):
    table = LOCAL_ROOT / "cell4" / exp["key"] / f"04_mcp_{exp['snap']}" / "mcp_state_and_locus.csv"
    if not table.exists():
        raise FileNotFoundError(
            f"Physical-distance Cell 4 table not found: {table}. "
            "Execute 01_live_cell_inhibitor_image_processing.ipynb first."
        )
    return table

def load_experiment(exp):
    source = DATA_ROOT / exp["rel"]
    table = find_cell4_table(exp)
    df = pd.read_csv(table, low_memory=False)
    required_distance_columns = {
        "mcp_tetr_distance_um", "mcp_tetr_distance_vox_index", "mcp_tetr_distance_unit"
    }
    missing_distance_columns = required_distance_columns.difference(df.columns)
    if missing_distance_columns:
        raise AssertionError(f"{exp['key']}: missing physical-distance columns {sorted(missing_distance_columns)}")
    observed_units = set(df.loc[df.mcp_tetr_distance_um.notna(), "mcp_tetr_distance_unit"].dropna())
    if observed_units and observed_units != {"um"}:
        raise AssertionError(f"{exp['key']}: unexpected MCP-mTetR distance units {observed_units}")
    df["experiment_key"] = exp["key"]
    df["date"] = exp["date"]
    df["replicate"] = exp["replicate"]
    df["gene"] = exp["gene"]
    df["snap"] = exp["snap"]
    df["display_snap"] = exp.get("display_snap", exp["snap"])
    df["design"] = exp["design"]
    qc_path = source / "notebook" / "outputs" / "06_qc" / "analysis_ready_primary.csv"
    if qc_path.exists():
        qc = pd.read_csv(qc_path, low_memory=False)
        qc_cols = [c for c in ["cell_id", "frame", "snap_mip_cnr", "snap_low_expression_cell", "any_saturated"] if c in qc.columns]
        if len(qc_cols) > 2:
            df = df.merge(qc[qc_cols].drop_duplicates(["cell_id", "frame"]), on=["cell_id", "frame"], how="left")
    if exp["design"] == "HDACi+THZ1":
        hdaci = df[df.group.isin(["DMSO", "TSA", "RGFP966"])].copy()
        hdaci["comparison"] = "HDACi"
        thz1 = df[df.group.isin(["DMSO", "THZ1"])].copy()
        thz1["comparison"] = "THZ1"
        thz1["replicate"] = exp["thz1_replicate"]
        df = pd.concat([hdaci, thz1], ignore_index=True)
    else:
        df["comparison"] = exp["design"]
    return df, table


## Dataset inventory

All Cell 4 tables are combined into one analysis table. The inventory records the source file, number of cells, and treatment groups for each experiment. Duplicate cell-frame rows and unexpected timepoints stop the analysis before correction or plotting.


In [ ]:
frames, inventory = [], []
for exp in EXPERIMENTS.values():
    df, table = load_experiment(exp)
    frames.append(df)
    counts = df.drop_duplicates(["cell_id", "group"]).groupby("group")["cell_id"].nunique().to_dict()
    inventory.append({**exp, "cell4_table": str(table), "rows": len(df.drop_duplicates(["cell_id", "frame", "group"])), "cells": df.cell_id.nunique(), "groups": "; ".join(f"{k}:{v}" for k, v in sorted(counts.items()))})

data = pd.concat(frames, ignore_index=True)
inventory = pd.DataFrame(inventory)
inventory.to_csv(TABLE_ROOT / "experiment_inventory.csv", index=False)

required_times = set(TIME_MIN)
problems = []
for (key, comparison), sub in data.groupby(["experiment_key", "comparison"]):
    if sub.duplicated(["cell_id", "frame"]).any():
        problems.append(f"{key}: duplicate cell/frame rows")
    if not set(sub.time_min.dropna().unique()).issubset(required_times):
        problems.append(f"{key}: unexpected timepoints")
if problems:
    raise AssertionError("; ".join(problems))

inventory[["key", "date", "replicate", "gene", "snap", "design", "cells", "groups"]]


## Matched-DMSO photobleaching correction

Local and nuclear signals are corrected separately because they have different spatial measurements and bleaching behavior. For each experiment, comparison, and signal definition, positive matched DMSO cells are divided by their own t0 value. The median DMSO trajectory is fitted with a t0-anchored log-quadratic curve constrained to be non-increasing.

The inverse fitted curve is applied to DMSO and every treatment in the same experiment. Treatment cells do not contribute to the fit, and DMSO residual variation is retained. Background-subtracted and total local and nuclear signals are processed separately. Local measurements require a reviewed mTetR coordinate and no analysis error; nuclear measurements require a valid cell mask.


In [ ]:
def signal_columns(snap):
    return {
        "local_bg": f"local_{snap}_background_subtracted_3d_mean_raw",
        "local_total": f"local_{snap}_total_3d_mean_raw",
        "nuclear_bg": f"nuclear_{snap}_background_subtracted_integrated_per_mask_area_raw",
        "nuclear_total": f"nuclear_{snap}_total_integrated_per_mask_area_raw",
    }

def fit_smooth_bleaching_curve(normalized):
    median = normalized.median(axis=0, skipna=True)
    reference_n = normalized.count(axis=0)
    if median.isna().any() or median.le(0).any():
        raise ValueError("Invalid DMSO reference")
    x = np.asarray(median.index, dtype=float) / max(TIME_MIN)
    y = np.log(median.to_numpy(dtype=float))
    weights = reference_n.to_numpy(dtype=float)
    weights = weights / weights.mean()
    objective = lambda beta: np.sum(weights * (y - (beta[0] * x + beta[1] * x ** 2)) ** 2)
    initial_slope = min(0.0, np.sum(weights[1:] * x[1:] * y[1:]) / np.sum(weights[1:] * x[1:] ** 2))
    constraints = [
        {"type": "ineq", "fun": lambda beta: -beta[0]},
        {"type": "ineq", "fun": lambda beta: -(beta[0] + 2 * beta[1])},
    ]
    fit = minimize(objective, [initial_slope, 0.0], method="SLSQP", constraints=constraints)
    if not fit.success:
        raise RuntimeError(f"Bleaching fit failed: {fit.message}")
    fitted_reference = pd.Series(np.exp(fit.x[0] * x + fit.x[1] * x ** 2), index=median.index)
    factor = 1.0 / fitted_reference
    factor.loc[0.0] = 1.0
    return reference_n, median, fitted_reference, factor, fit.x

def fit_dmso_reference(sub, raw_col, domain):
    valid = np.isfinite(pd.to_numeric(sub[raw_col], errors="coerce")) & (pd.to_numeric(sub[raw_col], errors="coerce") > 0)
    if domain == "local":
        valid &= sub.tetr_valid.fillna(False).astype(bool) & sub.review_status.eq("reviewed") & sub.analysis_error.fillna("").eq("")
    else:
        valid &= pd.to_numeric(sub.mask_area_px, errors="coerce").fillna(0).gt(0)
    dmso = sub[valid & sub.group.eq("DMSO")].pivot_table(index="cell_id", columns="time_min", values=raw_col, aggfunc="first").reindex(columns=TIME_MIN)
    dmso = dmso[dmso[0.0].gt(0)]
    if dmso.empty:
        raise ValueError(f"No positive DMSO t0 values for {raw_col}")
    normalized = dmso.div(dmso[0.0], axis=0)
    reference_n, median, fitted_reference, factor, coefficients = fit_smooth_bleaching_curve(normalized)
    return dmso.index, reference_n, median, fitted_reference, factor, coefficients

corrected_parts, factor_rows = [], []
for (key, comparison), sub in data.groupby(["experiment_key", "comparison"], sort=False):
    snap = sub.snap.iloc[0]
    for metric, raw_col in signal_columns(snap).items():
        if raw_col not in sub.columns:
            raise KeyError(f"{key}: missing {raw_col}")
        domain = "local" if metric.startswith("local") else "nuclear"
        cells, reference_n, median, fitted_reference, factor, coefficients = fit_dmso_reference(sub, raw_col, domain)
        part = sub[["experiment_key", "date", "replicate", "gene", "snap", "display_snap", "design", "comparison", "cell_id", "field_id", "group", "frame", "time_min", "state", "review_status", "tetr_valid", "mask_area_px", "analysis_error"]].copy()
        for optional in ["snap_mip_cnr", "snap_low_expression_cell", "any_saturated"]:
            if optional in sub.columns:
                part[optional] = sub[optional]
        part["metric"] = metric
        part["raw_column"] = raw_col
        part["raw_signal"] = pd.to_numeric(sub[raw_col], errors="coerce")
        part["correction_factor"] = part.time_min.map(factor)
        part["corrected_signal"] = part.raw_signal * part.correction_factor
        valid = np.isfinite(part.raw_signal) & part.raw_signal.gt(0)
        if domain == "local":
            valid &= part.tetr_valid.fillna(False).astype(bool) & part.review_status.eq("reviewed") & part.analysis_error.fillna("").eq("")
        else:
            valid &= pd.to_numeric(part.mask_area_px, errors="coerce").fillna(0).gt(0)
        part["valid_primary"] = valid
        t0 = part[part.time_min.eq(0) & part.valid_primary][["cell_id", "corrected_signal"]].rename(columns={"corrected_signal": "t0"})
        part = part.merge(t0, on="cell_id", how="left")
        part["fold_change"] = np.where(part.valid_primary & part.t0.gt(0), part.corrected_signal / part.t0, np.nan)
        corrected_parts.append(part)
        for time in TIME_MIN:
            factor_rows.append(dict(experiment_key=key, comparison=comparison, metric=metric, time_min=time, dmso_t0_positive_cells=len(cells), dmso_reference_n=int(reference_n.loc[time]), dmso_raw_median_ratio=median.loc[time], fitted_bleaching_reference=fitted_reference.loc[time], correction_factor=factor.loc[time], log_quadratic_linear_coefficient=coefficients[0], log_quadratic_squared_coefficient=coefficients[1], dmso_model_residual_log2=np.log2(median.loc[time] / fitted_reference.loc[time]), factor_outside_0_5_to_2=bool(factor.loc[time] < 0.5 or factor.loc[time] > 2.0)))

corrected = pd.concat(corrected_parts, ignore_index=True)
factors = pd.DataFrame(factor_rows)
factors.to_csv(TABLE_ROOT / "bleaching_correction_factors.csv", index=False)

summary = corrected.groupby(["experiment_key", "date", "replicate", "gene", "display_snap", "design", "comparison", "metric", "group", "time_min"], dropna=False).agg(
    n=("fold_change", "count"),
    median_fold_change=("fold_change", "median"),
    mean_fold_change=("fold_change", "mean"),
    sem_fold_change=("fold_change", lambda x: x.std(ddof=1) / math.sqrt(x.count()) if x.count() > 1 else np.nan),
).reset_index()
summary.to_csv(TABLE_ROOT / "signal_trajectories.csv", index=False)

dmso = summary[summary.group.eq("DMSO")][["experiment_key", "comparison", "metric", "time_min", "median_fold_change"]].rename(columns={"median_fold_change": "dmso_median_fold_change"})
did = summary.merge(dmso, on=["experiment_key", "comparison", "metric", "time_min"], how="left")
did["treatment_minus_dmso"] = did.median_fold_change - did.dmso_median_fold_change
did.to_csv(TABLE_ROOT / "treatment_minus_dmso.csv", index=False)

state = data[data.state.isin(["Active", "Inactive"])].copy()
active = state.groupby(["experiment_key", "date", "replicate", "gene", "display_snap", "design", "comparison", "group", "time_min"]).state.agg(n="count", active_n=lambda x: (x == "Active").sum()).reset_index()
active["active_fraction"] = active.active_n / active.n
active.to_csv(TABLE_ROOT / "active_fraction.csv", index=False)

qc = corrected[corrected.metric.eq("local_bg")].groupby(["experiment_key", "comparison", "group", "time_min"]).agg(
    total_rows=("cell_id", "size"),
    valid_local_n=("valid_primary", "sum"),
    positive_signal_n=("raw_signal", lambda x: (x > 0).sum()),
    cnr_ge_1_n=("snap_mip_cnr", lambda x: (x >= 1).sum()) if "snap_mip_cnr" in corrected.columns else ("cell_id", lambda x: np.nan),
).reset_index()
qc.to_csv(TABLE_ROOT / "qc_counts.csv", index=False)

summary.head()


## Per-cell normalization and statistics

Each corrected signal is divided by the same cell's corrected t0 value and reported as `log2(signal / t0)`. Replicates remain separate. Lines show the median across cells and bands show the interquartile range; the mean and SEM are also saved in the tables.

Local and nuclear signal distributions are compared at 15, 30, 45, and 60 min with two-sided Mann-Whitney U tests. Holm correction is applied within each replicate, comparison branch, and signal metric. Figures show only significant treatment-versus-DMSO results. Transcriptional Active fractions are descriptive and are not tested for significance. Shapiro-Wilk results are saved as a distribution check, not as a rule for choosing the primary test.

Matched DMSO flatness is also checked by leaving out one imaging field at a time, fitting the correction to the remaining fields, and applying it to the held-out field.


In [ ]:
from itertools import combinations
from scipy.stats import mannwhitneyu, shapiro

PRIMARY_METRICS = ("local_bg", "nuclear_bg")
GROUP_ORDER = ("DMSO", "TSA", "RGFP966", "THZ1")
corrected["log2_fold_change"] = np.where(
    corrected.valid_primary & corrected.fold_change.gt(0),
    np.log2(corrected.fold_change),
    np.nan,
)

per_cell_columns = [
    "experiment_key", "date", "replicate", "gene", "display_snap",
    "comparison", "cell_id", "field_id", "group", "time_min", "metric",
    "raw_signal", "correction_factor", "corrected_signal",
    "fold_change", "log2_fold_change", "valid_primary",
]
for optional in ["snap_mip_cnr", "snap_low_expression_cell", "any_saturated"]:
    if optional in corrected.columns:
        per_cell_columns.append(optional)
corrected[corrected.metric.isin(PRIMARY_METRICS)][per_cell_columns].to_csv(
    TABLE_ROOT / "per_cell_log2_fold_change.csv", index=False
)

signal_group_columns = [
    "experiment_key", "date", "replicate", "gene", "display_snap",
    "comparison", "metric", "group", "time_min",
]
signal_summary = corrected[corrected.metric.isin(PRIMARY_METRICS)].groupby(
    signal_group_columns, dropna=False
).agg(
    n=("log2_fold_change", "count"),
    median_log2_fc=("log2_fold_change", "median"),
    q25_log2_fc=("log2_fold_change", lambda x: x.quantile(0.25)),
    q75_log2_fc=("log2_fold_change", lambda x: x.quantile(0.75)),
    mean_log2_fc=("log2_fold_change", "mean"),
    sem_log2_fc=("log2_fold_change", lambda x: x.std(ddof=1) / math.sqrt(x.count()) if x.count() > 1 else np.nan),
).reset_index()
signal_summary.to_csv(TABLE_ROOT / "signal_trajectories_log2.csv", index=False)

normality_rows = []
for keys, sub in corrected[corrected.metric.isin(PRIMARY_METRICS) & corrected.time_min.gt(0)].groupby(signal_group_columns):
    values = sub.log2_fold_change.dropna().to_numpy()
    p_value = shapiro(values).pvalue if 3 <= len(values) <= 5000 else np.nan
    normality_rows.append(dict(zip(signal_group_columns, keys), n=len(values), shapiro_p=p_value, skew=pd.Series(values).skew()))
normality = pd.DataFrame(normality_rows)
normality["reject_normality_0_05"] = normality.shapiro_p.lt(0.05)
normality.to_csv(TABLE_ROOT / "normality_log2_fold_change.csv", index=False)

def holm_adjust(values):
    values = np.asarray(values, dtype=float)
    adjusted = np.full(values.shape, np.nan)
    finite = np.flatnonzero(np.isfinite(values))
    order = finite[np.argsort(values[finite])]
    running = 0.0
    total = len(order)
    for rank, index in enumerate(order):
        running = max(running, min(1.0, (total - rank) * values[index]))
        adjusted[index] = running
    return adjusted

signal_test_rows = []
test_columns = ["experiment_key", "date", "replicate", "gene", "display_snap", "comparison", "metric"]
for keys, sub in corrected[corrected.metric.isin(PRIMARY_METRICS)].groupby(test_columns):
    groups = [group for group in GROUP_ORDER if group in set(sub.group)]
    for time in TIME_MIN[1:]:
        at_time = sub[sub.time_min.eq(time)]
        for group_1, group_2 in combinations(groups, 2):
            values_1 = at_time.loc[at_time.group.eq(group_1), "log2_fold_change"].dropna().to_numpy()
            values_2 = at_time.loc[at_time.group.eq(group_2), "log2_fold_change"].dropna().to_numpy()
            p_value = mannwhitneyu(values_1, values_2, alternative="two-sided").pvalue if len(values_1) and len(values_2) else np.nan
            signal_test_rows.append(dict(
                zip(test_columns, keys), time_min=time, group_1=group_1, group_2=group_2,
                n_1=len(values_1), n_2=len(values_2),
                median_1=np.median(values_1) if len(values_1) else np.nan,
                median_2=np.median(values_2) if len(values_2) else np.nan,
                median_difference_log2=(np.median(values_2) - np.median(values_1)) if len(values_1) and len(values_2) else np.nan,
                mann_whitney_p=p_value,
            ))
signal_tests = pd.DataFrame(signal_test_rows)
signal_tests["holm_p"] = signal_tests.groupby(
    ["experiment_key", "comparison", "metric"]
)["mann_whitney_p"].transform(holm_adjust)
signal_tests.to_csv(TABLE_ROOT / "signal_pairwise_tests.csv", index=False)

dmso_flatness = signal_summary[signal_summary.group.eq("DMSO")][
    ["experiment_key", "comparison", "metric", "time_min", "n", "median_log2_fc"]
].copy()
dmso_flatness["absolute_median_log2_fc"] = dmso_flatness.median_log2_fc.abs()
dmso_flatness.to_csv(TABLE_ROOT / "dmso_flatness_validation.csv", index=False)
dmso_flatness["within_0_20_log2"] = dmso_flatness.absolute_median_log2_fc.le(0.20)

cross_validation_rows = []
cross_validation_keys = ["experiment_key", "comparison", "metric"]
dmso_primary = corrected[corrected.metric.isin(PRIMARY_METRICS) & corrected.group.eq("DMSO") & corrected.valid_primary].copy()
for keys, sub in dmso_primary.groupby(cross_validation_keys):
    fields = sorted(sub.field_id.dropna().unique())
    if len(fields) < 2:
        continue
    for held_out_field in fields:
        training = sub[~sub.field_id.eq(held_out_field)].pivot_table(index="cell_id", columns="time_min", values="raw_signal", aggfunc="first").reindex(columns=TIME_MIN)
        training = training[training[0.0].gt(0)]
        if training.empty:
            continue
        training_normalized = training.div(training[0.0], axis=0)
        _, _, _, factor, _ = fit_smooth_bleaching_curve(training_normalized)
        held_out = sub[sub.field_id.eq(held_out_field)].pivot_table(index="cell_id", columns="time_min", values="raw_signal", aggfunc="first").reindex(columns=TIME_MIN)
        held_out = held_out[held_out[0.0].gt(0)]
        held_out_log2 = np.log2(held_out.div(held_out[0.0], axis=0).mul(factor, axis=1))
        for cell_id, values in held_out_log2.iterrows():
            for time, value in values.items():
                cross_validation_rows.append(dict(zip(cross_validation_keys, keys), held_out_field=held_out_field, cell_id=cell_id, time_min=time, log2_fold_change=value))
dmso_cross_validation = pd.DataFrame(cross_validation_rows)
dmso_cross_validation.to_csv(TABLE_ROOT / "dmso_leave_one_field_out_per_cell.csv", index=False)
dmso_cross_validation_summary = dmso_cross_validation.groupby(cross_validation_keys + ["time_min"]).log2_fold_change.agg(
    n="count", median_log2_fc="median", q25_log2_fc=lambda x: x.quantile(0.25), q75_log2_fc=lambda x: x.quantile(0.75)
).reset_index()
dmso_cross_validation_summary["absolute_median_log2_fc"] = dmso_cross_validation_summary.median_log2_fc.abs()
dmso_cross_validation_summary.to_csv(TABLE_ROOT / "dmso_leave_one_field_out_summary.csv", index=False)

print(f"Shapiro-Wilk rejected normality in {normality.reject_normality_0_05.sum()} of {normality.shapiro_p.notna().sum()} distributions.")


## Raw-signal audit

Uncorrected measurements are saved in a separate audit folder. The exports include one combined per-cell table, one combined summary, and separate files and plots for every replicate and comparison branch. Total and background-subtracted local and nuclear signals are kept so that the correction can be checked against the source measurements.


In [ ]:
RAW_AUDIT_ROOT = RESULT_ROOT / "raw_signal_audit"
RAW_PER_CELL_ROOT = RAW_AUDIT_ROOT / "per_cell"
RAW_SUMMARY_ROOT = RAW_AUDIT_ROOT / "summaries"
RAW_FIGURE_ROOT = RAW_AUDIT_ROOT / "figures"
for directory in [RAW_PER_CELL_ROOT, RAW_SUMMARY_ROOT, RAW_FIGURE_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

raw_columns = [
    "experiment_key", "date", "replicate", "gene", "display_snap", "comparison",
    "cell_id", "field_id", "group", "frame", "time_min", "state", "metric",
    "raw_column", "raw_signal", "valid_primary",
]
for optional in ["snap_mip_cnr", "snap_low_expression_cell", "any_saturated"]:
    if optional in corrected.columns:
        raw_columns.append(optional)
raw_per_cell = corrected[raw_columns].copy()
raw_per_cell.to_csv(RAW_PER_CELL_ROOT / "all_experiments_raw_signal_per_cell.csv", index=False)
raw_summary_columns = ["experiment_key", "date", "replicate", "gene", "display_snap", "comparison", "metric", "group", "time_min"]
raw_summary = raw_per_cell[raw_per_cell.valid_primary].groupby(raw_summary_columns).raw_signal.agg(
    n="count", median_raw="median", q25_raw=lambda x: x.quantile(0.25), q75_raw=lambda x: x.quantile(0.75), mean_raw="mean", sem_raw=lambda x: x.std(ddof=1) / math.sqrt(x.count()) if x.count() > 1 else np.nan
).reset_index()
raw_summary.to_csv(RAW_SUMMARY_ROOT / "all_experiments_raw_signal_summary.csv", index=False)

raw_branch_manifest = []
for (key, comparison), branch in raw_per_cell.groupby(["experiment_key", "comparison"], sort=True):
    meta = branch.iloc[0]
    stem = f"{comparison}_{meta.gene}_{meta.display_snap}_rep{int(meta.replicate)}_{meta.date}"
    per_cell_path = RAW_PER_CELL_ROOT / f"{stem}_raw_per_cell.csv"
    summary_path = RAW_SUMMARY_ROOT / f"{stem}_raw_summary.csv"
    branch.to_csv(per_cell_path, index=False)
    raw_summary[raw_summary.experiment_key.eq(key) & raw_summary.comparison.eq(comparison)].to_csv(summary_path, index=False)
    raw_branch_manifest.append(dict(experiment_key=key, comparison=comparison, per_cell_csv=str(per_cell_path), summary_csv=str(summary_path)))
pd.DataFrame(raw_branch_manifest).to_csv(RAW_AUDIT_ROOT / "raw_signal_manifest.csv", index=False)
(RAW_AUDIT_ROOT / "README.md").write_text("# Raw signal audit\n\nAll values in this folder are upstream of smooth DMSO-derived photobleaching correction. `per_cell` contains individual observations; `summaries` contains n, median, interquartile range, mean, and SEM; `figures` contains replicate-level raw median and IQR plots. Background-subtracted and total local/nuclear definitions are retained separately.\n", encoding="utf-8")


## Figure settings

Main figures use a 120 by 60 mm two-panel layout. Supplementary figures use a 60 by 60 mm single-panel layout. Active fractions are shown with 95% Wilson intervals. Signal panels show median `log2(signal / t0)` with interquartile ranges, and significant adjusted p values are displayed as asterisks. Raw audit plots are saved at 600 dpi.


In [ ]:
MAIN_FIGURE_ROOT = FIGURE_ROOT / "main_by_replicate"
SUPPLEMENT_FIGURE_ROOT = FIGURE_ROOT / "supplement_by_replicate"
for directory in [MAIN_FIGURE_ROOT, SUPPLEMENT_FIGURE_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)
MAIN_FIGURE_SIZE_INCHES = (120 / 25.4, 60 / 25.4)
SUPPLEMENT_FIGURE_SIZE_INCHES = (60 / 25.4, 60 / 25.4)

def wilson_interval(active_n, total_n, z=1.96):
    proportion = active_n / total_n
    denominator = 1 + z ** 2 / total_n
    center = (proportion + z ** 2 / (2 * total_n)) / denominator
    half_width = z * math.sqrt(proportion * (1 - proportion) / total_n + z ** 2 / (4 * total_n ** 2)) / denominator
    return center - half_width, center + half_width

active[["ci_low", "ci_high"]] = active.apply(
    lambda row: pd.Series(wilson_interval(row.active_n, row.n)), axis=1
)

def p_symbol(p_value):
    if not np.isfinite(p_value):
        return ""
    if p_value < 0.0001:
        return "****"
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return ""

def signal_limits(comparison, target, metric):
    subset = signal_summary[
        signal_summary.comparison.eq(comparison)
        & signal_summary.display_snap.eq(target)
        & signal_summary.metric.eq(metric)
    ]
    low = min(-0.25, subset.q25_log2_fc.min())
    high = max(0.25, subset.q75_log2_fc.max())
    span = max(1.0, high - low)
    annotation_rows = 2 if comparison == "HDACi" else 1
    low = math.floor((low - 0.08 * span) * 2) / 2
    high = math.ceil((high + (0.16 + 0.10 * annotation_rows) * span) * 2) / 2
    return low, high

def annotate_vs_dmso(ax, tests, treatments, include_t0):
    low, high = ax.get_ylim()
    span = high - low
    times = TIME_MIN if include_t0 else TIME_MIN[1:]
    for row_index, treatment in enumerate(treatments):
        y = high - (0.10 + 0.09 * row_index) * span
        for time in times:
            match = tests[
                tests.time_min.eq(time)
                & tests.group_1.eq("DMSO")
                & tests.group_2.eq(treatment)
            ]
            if not match.empty and match.holm_p.iloc[0] < 0.05:
                ax.text(time, y, p_symbol(match.holm_p.iloc[0]), color=COLORS[treatment], ha="center", va="center", fontsize=5.5)


def plot_active_panel(ax, subset, groups, gene):
    for group in groups:
        values = subset[subset.group.eq(group)].sort_values("time_min")
        x = values.time_min.to_numpy(dtype=float)
        y = values.active_fraction.to_numpy(dtype=float)
        ax.fill_between(x, values.ci_low.to_numpy(dtype=float), values.ci_high.to_numpy(dtype=float), color=COLORS[group], alpha=0.12, linewidth=0)
        ax.plot(x, y, color=COLORS[group], marker="o", label=group)
    ax.set(xticks=TIME_MIN, ylim=(-0.04, 1.04), xlabel="Time (min)", ylabel="Active fraction", title=rf"$\it{{{gene}}}$ transcription")
    style_axis(ax)

def plot_signal_panel(ax, subset, tests, groups, comparison, target, metric, title):
    for group in groups:
        values = subset[subset.group.eq(group)].sort_values("time_min")
        x = values.time_min.to_numpy(dtype=float)
        y = values.median_log2_fc.to_numpy(dtype=float)
        ax.fill_between(x, values.q25_log2_fc.to_numpy(dtype=float), values.q75_log2_fc.to_numpy(dtype=float), color=COLORS[group], alpha=0.12, linewidth=0)
        ax.plot(x, y, color=COLORS[group], marker="o", label=group)
    ax.axhline(0, color="0.70", linewidth=0.6, zorder=0)
    if "$\\it{" not in title:
        gene = subset.gene.iloc[0]
        title = title.replace(gene, rf"$\it{{{gene}}}$", 1)
    ax.set(xticks=TIME_MIN, ylim=signal_limits(comparison, target, metric), xlabel="Time (min)", ylabel=r"Relative SNAP intensity (log$_2$[$I_t$/$I_0$])", title=title)
    style_axis(ax)
    annotate_vs_dmso(ax, tests, [group for group in groups if group != "DMSO"], include_t0=False)


def save_replicate_figure(fig, output_root, stem):
    png_path = output_root / f"{stem}.png"
    pdf_path = output_root / f"{stem}.pdf"
    fig.savefig(png_path, dpi=600)
    fig.savefig(pdf_path)
    plt.close(fig)
    return png_path, pdf_path

def plot_raw_audit(branch_summary, groups, gene, target, stem):
    metric_titles = [("local_bg", f"{gene} local background-subtracted"), ("local_total", f"{gene} local total"), ("nuclear_bg", f"{gene} nuclear background-subtracted"), ("nuclear_total", f"{gene} nuclear total")]
    fig, axes = plt.subplots(2, 2, figsize=(180 / 25.4, 100 / 25.4), sharex=True)
    for ax, (metric, title) in zip(axes.flat, metric_titles):
        metric_data = branch_summary[branch_summary.metric.eq(metric)]
        for group in groups:
            values = metric_data[metric_data.group.eq(group)].sort_values("time_min")
            x = values.time_min.to_numpy(dtype=float)
            ax.fill_between(x, values.q25_raw.to_numpy(dtype=float), values.q75_raw.to_numpy(dtype=float), color=COLORS[group], alpha=0.12, linewidth=0)
            ax.plot(x, values.median_raw.to_numpy(dtype=float), color=COLORS[group], marker="o", label=group)
        ax.set(xticks=TIME_MIN, xlabel="Time (min)", ylabel="Raw intensity (a.u.)", title=title)
        style_axis(ax)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.995), ncol=len(groups), handlelength=1.6, columnspacing=1.0)
    fig.subplots_adjust(left=0.085, right=0.985, bottom=0.14, top=0.84, wspace=0.34, hspace=0.52)
    output_path = RAW_FIGURE_ROOT / f"{stem}_raw_signals.png"
    fig.savefig(output_path, dpi=600)
    plt.close(fig)
    return output_path

## Replicate figures

All datasets remain in the numerical tables and statistical analyses. For figure-file generation, a series with more than two acquisition datasets is represented by the two treatment-minus-DMSO local-signal trajectories closest to the coordinate-wise median trajectory. The mean absolute deviation score and display decision are saved in `figure_replicate_selection.csv`; no observations are removed from the underlying analysis tables.

The main figure contains transcriptional Active fraction and gene-specific local SNAP. The supplementary figure contains nuclear SNAP. Internal identifiers are retained in filenames but are not repeated inside the plot. No significance symbols are added to transcription. Local and nuclear panels show symbols only for significant treatment-versus-DMSO comparisons after Holm correction; `ns` labels are omitted.

In [ ]:
def build_figure_replicate_selection(signal_summary):
    series_columns = ["comparison", "gene", "display_snap"]
    local = signal_summary[
        signal_summary.metric.eq("local_bg") & signal_summary.time_min.gt(0)
    ].copy()
    selection_rows = []
    for series_keys, series in local.groupby(series_columns, sort=True):
        metadata = dict(zip(series_columns, series_keys))
        experiment_rows = series[["experiment_key", "date", "replicate"]].drop_duplicates()
        experiment_rows = experiment_rows.sort_values(["date", "experiment_key"])
        treatments = [group for group in GROUP_ORDER if group != "DMSO" and group in set(series.group)]
        coordinates = [(group, time) for group in treatments for time in TIME_MIN[1:]]
        vectors = []
        for row in experiment_rows.itertuples(index=False):
            experiment = series[series.experiment_key.eq(row.experiment_key)]
            lookup = experiment.set_index(["group", "time_min"])["median_log2_fc"]
            values = []
            for group, time in coordinates:
                dmso_value = lookup.get(("DMSO", time), np.nan)
                treatment_value = lookup.get((group, time), np.nan)
                values.append(treatment_value - dmso_value)
            vectors.append(np.asarray(values, dtype=float))
        matrix = np.vstack(vectors)
        robust_center = np.nanmedian(matrix, axis=0)
        scores = np.asarray([
            np.nanmean(np.abs(vector - robust_center)) if np.isfinite(vector - robust_center).any() else np.inf
            for vector in matrix
        ])
        selected_count = min(2, len(experiment_rows))
        selected_indices = set(np.argsort(scores, kind="stable")[:selected_count].tolist())
        for index, row in enumerate(experiment_rows.itertuples(index=False)):
            selection_rows.append({
                **metadata,
                "experiment_key": row.experiment_key,
                "date": row.date,
                "replicate": row.replicate,
                "trajectory_coordinates": len(coordinates),
                "finite_coordinates": int(np.isfinite(matrix[index]).sum()),
                "robust_center_mean_absolute_deviation": float(scores[index]),
                "selected_for_figure": index in selected_indices,
                "selection_scope": "figure files only; all datasets retained in numerical tables",
            })
    return pd.DataFrame(selection_rows)


figure_selection = build_figure_replicate_selection(signal_summary)
figure_selection.to_csv(TABLE_ROOT / "figure_replicate_selection.csv", index=False)
selected_figure_branches = set(
    map(tuple, figure_selection.loc[figure_selection.selected_for_figure, ["experiment_key", "comparison"]].to_numpy())
)

manifest_rows = []
replicate_keys = ["experiment_key", "date", "replicate", "gene", "display_snap", "comparison"]
for keys, replicate_signal in signal_summary.groupby(replicate_keys, sort=True):
    metadata = dict(zip(replicate_keys, keys))
    key = metadata["experiment_key"]
    comparison = metadata["comparison"]
    if (key, comparison) not in selected_figure_branches:
        continue
    target = metadata["display_snap"]
    groups = [group for group in GROUP_ORDER if group in set(replicate_signal.group)]
    replicate_active = active[active.experiment_key.eq(key) & active.comparison.eq(comparison)]
    replicate_signal_tests = signal_tests[signal_tests.experiment_key.eq(key) & signal_tests.comparison.eq(comparison)]

    main_fig, main_axes = plt.subplots(1, 2, figsize=MAIN_FIGURE_SIZE_INCHES)
    plot_active_panel(main_axes[0], replicate_active, groups, metadata["gene"])
    plot_signal_panel(main_axes[1], replicate_signal[replicate_signal.metric.eq("local_bg")], replicate_signal_tests[replicate_signal_tests.metric.eq("local_bg")], groups, comparison, target, "local_bg", f"{metadata['gene']} local {target}")
    for label, ax in zip("ab", main_axes):
        ax.text(0.01, 1.08, label, transform=ax.transAxes, fontweight="bold", fontsize=8, ha="left")
    handles, labels = main_axes[0].get_legend_handles_labels()
    main_fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.995), ncol=len(groups), handlelength=1.6, columnspacing=1.0)
    main_fig.subplots_adjust(left=0.105, right=0.98, bottom=0.22, top=0.78, wspace=0.42)
    safe_target = target.replace(" ", "_")
    stem = f"{comparison}_{metadata['gene']}_{safe_target}_rep{int(metadata['replicate'])}_{metadata['date']}"
    main_png, main_pdf = save_replicate_figure(main_fig, MAIN_FIGURE_ROOT, f"main_{stem}")

    supplement_fig, supplement_ax = plt.subplots(1, 1, figsize=SUPPLEMENT_FIGURE_SIZE_INCHES)
    plot_signal_panel(supplement_ax, replicate_signal[replicate_signal.metric.eq("nuclear_bg")], replicate_signal_tests[replicate_signal_tests.metric.eq("nuclear_bg")], groups, comparison, target, "nuclear_bg", f"{metadata['gene']} nuclear {target}")
    handles, labels = supplement_ax.get_legend_handles_labels()
    supplement_fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.995), ncol=min(len(groups), 3), handlelength=1.6, columnspacing=1.0)
    supplement_fig.subplots_adjust(left=0.22, right=0.96, bottom=0.22, top=0.78)
    supplement_png, supplement_pdf = save_replicate_figure(supplement_fig, SUPPLEMENT_FIGURE_ROOT, f"supplement_{stem}")

    branch_raw_summary = raw_summary[raw_summary.experiment_key.eq(key) & raw_summary.comparison.eq(comparison)]
    raw_figure = plot_raw_audit(branch_raw_summary, groups, metadata["gene"], target, stem)
    manifest_rows.append({**metadata, "groups": ";".join(groups), "qc_limited": target == "HDAC3", "main_png": str(main_png), "main_pdf": str(main_pdf), "supplement_png": str(supplement_png), "supplement_pdf": str(supplement_pdf), "raw_audit_png": str(raw_figure)})

figure_manifest = pd.DataFrame(manifest_rows)
figure_manifest.to_csv(TABLE_ROOT / "replicate_figure_manifest.csv", index=False)
expected_figure_count = int(figure_selection.selected_for_figure.sum())
if len(figure_manifest) != expected_figure_count:
    raise AssertionError(f"Expected {expected_figure_count} selected figure branches, found {len(figure_manifest)}")
figure_selection.sort_values(["comparison", "gene", "display_snap", "selected_for_figure", "robust_center_mean_absolute_deviation"])


## Manuscript figure exports

This cell exports the selected panels for Supplementary Fig. 13 and the selected
THZ1 components for the main figure. Output folders are derived from
`LOCAL_ROOT`, whose default is resolved relative to the notebook working
directory. Set `LIVE_INHIBITOR_OUTPUT_ROOT` to override that default.

Supplementary Fig. 13a and 13b each show a single pooled-cell trajectory formed
from the specified HDACi rep1 and rep2 experiments. Cell-level values retain the
experiment-specific t0 normalization and matched-DMSO photobleaching correction;
the median, interquartile range, Mann-Whitney tests, and Holm correction are then
recalculated after pooling the two experiments. These panels are pooled-cell
visualizations and do not model biological replicate as a random effect. The
main-figure components retain the two-panel transcription/local-SNAP layout but
omit internal panel letters so that lettering can be added during final figure
assembly.


In [ ]:
MANUSCRIPT_EXPORT_ROOT = RESULT_ROOT / "manuscript_figure_exports"
SUPP13_EXPORT_ROOT = MANUSCRIPT_EXPORT_ROOT / "Supplementary_Fig_13"
MAIN_COMPONENT_EXPORT_ROOT = MANUSCRIPT_EXPORT_ROOT / "Main_Figure_components"
for directory in [MANUSCRIPT_EXPORT_ROOT, SUPP13_EXPORT_ROOT, MAIN_COMPONENT_EXPORT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)


def manuscript_branch(experiment_key, comparison):
    branch_signal = signal_summary[
        signal_summary.experiment_key.eq(experiment_key)
        & signal_summary.comparison.eq(comparison)
    ].copy()
    if branch_signal.empty:
        raise KeyError(f"No signal summary for {experiment_key}, {comparison}")

    metadata = branch_signal.iloc[0]
    target = metadata.display_snap
    groups = [group for group in GROUP_ORDER if group in set(branch_signal.group)]
    branch_tests = signal_tests[
        signal_tests.experiment_key.eq(experiment_key)
        & signal_tests.comparison.eq(comparison)
    ].copy()
    branch_active = active[
        active.experiment_key.eq(experiment_key)
        & active.comparison.eq(comparison)
    ].copy()
    return branch_signal, branch_tests, branch_active, metadata, target, groups


def save_manuscript_figure(fig, output_root, stem):
    png_path = output_root / f"{stem}.png"
    pdf_path = output_root / f"{stem}.pdf"
    fig.savefig(png_path, dpi=600)
    fig.savefig(pdf_path)
    plt.close(fig)
    return png_path, pdf_path


def pooled_manuscript_local_branch(experiment_keys, comparison):
    if len(experiment_keys) != 2:
        raise ValueError("A Supplementary Fig. 13 pooled panel requires rep1 and rep2")

    branch = corrected[
        corrected.experiment_key.isin(experiment_keys)
        & corrected.comparison.eq(comparison)
        & corrected.metric.eq("local_bg")
    ].copy()
    if branch.empty:
        raise KeyError(
            f"No pooled local signal for {experiment_keys}, {comparison}"
        )

    observed_keys = set(branch.experiment_key.unique())
    missing_keys = set(experiment_keys).difference(observed_keys)
    if missing_keys:
        raise KeyError(f"Missing experiments from pooled panel: {sorted(missing_keys)}")

    metadata = branch[["gene", "display_snap"]].drop_duplicates()
    if len(metadata) != 1:
        raise AssertionError(
            "Pooled Supplementary Fig. 13 inputs must share one gene and SNAP target"
        )
    gene = str(metadata.gene.iloc[0])
    target = str(metadata.display_snap.iloc[0])
    groups = [group for group in GROUP_ORDER if group in set(branch.group)]

    pooled_summary = branch.groupby(["group", "time_min"], dropna=False).agg(
        n=("log2_fold_change", "count"),
        median_log2_fc=("log2_fold_change", "median"),
        q25_log2_fc=("log2_fold_change", lambda x: x.quantile(0.25)),
        q75_log2_fc=("log2_fold_change", lambda x: x.quantile(0.75)),
        mean_log2_fc=("log2_fold_change", "mean"),
        sem_log2_fc=(
            "log2_fold_change",
            lambda x: x.std(ddof=1) / math.sqrt(x.count())
            if x.count() > 1 else np.nan,
        ),
    ).reset_index()
    pooled_summary["gene"] = gene
    pooled_summary["display_snap"] = target
    pooled_summary["comparison"] = comparison
    pooled_summary["metric"] = "local_bg"

    test_rows = []
    for time in TIME_MIN[1:]:
        at_time = branch[branch.time_min.eq(time)]
        for group_1, group_2 in combinations(groups, 2):
            values_1 = at_time.loc[
                at_time.group.eq(group_1), "log2_fold_change"
            ].dropna().to_numpy()
            values_2 = at_time.loc[
                at_time.group.eq(group_2), "log2_fold_change"
            ].dropna().to_numpy()
            p_value = (
                mannwhitneyu(values_1, values_2, alternative="two-sided").pvalue
                if len(values_1) and len(values_2)
                else np.nan
            )
            test_rows.append({
                "time_min": time,
                "group_1": group_1,
                "group_2": group_2,
                "n_1": len(values_1),
                "n_2": len(values_2),
                "mann_whitney_p": p_value,
            })
    pooled_tests = pd.DataFrame(test_rows)
    pooled_tests["holm_p"] = holm_adjust(pooled_tests.mann_whitney_p)
    return pooled_summary, pooled_tests, gene, target, groups


def export_supplement_local_pooled(experiment_keys, comparison, stem):
    pooled_summary, pooled_tests, gene, target, groups = (
        pooled_manuscript_local_branch(experiment_keys, comparison)
    )
    fig, ax = plt.subplots(1, 1, figsize=SUPPLEMENT_FIGURE_SIZE_INCHES)
    plot_signal_panel(
        ax,
        pooled_summary,
        pooled_tests,
        groups,
        comparison,
        target,
        "local_bg",
        rf"$\it{{{gene}}}$ local {target}",
    )

    handles, labels = ax.get_legend_handles_labels()
    handle_lookup = dict(zip(labels, handles))
    ordered_groups = [group for group in GROUP_ORDER if group in handle_lookup]
    fig.legend(
        [handle_lookup[group] for group in ordered_groups],
        ordered_groups,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=len(ordered_groups),
        handlelength=1.6,
        columnspacing=1.0,
    )
    fig.subplots_adjust(left=0.22, right=0.96, bottom=0.32, top=0.78)
    png_path, pdf_path = save_manuscript_figure(fig, SUPP13_EXPORT_ROOT, stem)
    return {
        "figure_role": stem.split("_")[1],
        "experiment_keys": ";".join(experiment_keys),
        "comparison": comparison,
        "genes": gene,
        "targets": target,
        "source_panels": ";".join(
            f"{key}:{comparison}:local_bg" for key in experiment_keys
        ),
        "analysis_level": "rep1+rep2 pooled cell-level values",
        "panel_letters": False,
        "png_path": str(png_path),
        "pdf_path": str(pdf_path),
    }


def export_main_figure_component(experiment_key, comparison, stem):
    branch_signal, branch_tests, branch_active, metadata, target, groups = manuscript_branch(
        experiment_key, comparison
    )
    fig, axes = plt.subplots(1, 2, figsize=MAIN_FIGURE_SIZE_INCHES)
    plot_active_panel(axes[0], branch_active, groups, metadata.gene)
    plot_signal_panel(
        axes[1],
        branch_signal[branch_signal.metric.eq("local_bg")],
        branch_tests[branch_tests.metric.eq("local_bg")],
        groups,
        comparison,
        target,
        "local_bg",
        f"{metadata.gene} local {target}",
    )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=len(groups),
        handlelength=1.6,
        columnspacing=1.0,
    )
    fig.subplots_adjust(left=0.105, right=0.98, bottom=0.22, top=0.78, wspace=0.42)
    png_path, pdf_path = save_manuscript_figure(
        fig, MAIN_COMPONENT_EXPORT_ROOT, stem
    )
    return {
        "figure_role": "main_figure_component",
        "experiment_keys": experiment_key,
        "comparison": comparison,
        "genes": str(metadata.gene),
        "targets": str(target),
        "source_panels": f"{experiment_key}:{comparison}:active_fraction+local_bg",
        "panel_letters": False,
        "png_path": str(png_path),
        "pdf_path": str(pdf_path),
    }


manuscript_export_rows = [
    export_supplement_local_pooled(
        ["260717_Nanog_H3K27ac", "260727_Nanog_H3K27ac"],
        "HDACi",
        "Fig_S13a_Nanog_HDACi_H3K27ac_local",
    ),
    export_supplement_local_pooled(
        ["260723_Sox2_H3K27ac", "260727_Sox2_H3K27ac"],
        "HDACi",
        "Fig_S13b_Sox2_HDACi_H3K27ac_local",
    ),
]

main_component_specs = [
    ("260727_Nanog_H3K27ac", "THZ1", "main_THZ1_Nanog_H3K27ac_rep1_260727"),
    ("260805_Sox2_H3K27ac", "THZ1", "main_THZ1_Sox2_H3K27ac_rep2_260805"),
    ("260807_Nanog_HDAC1", "THZ1", "main_THZ1_Nanog_HDAC1_rep3_260807"),
    ("260806_Nanog_HDAC3", "THZ1", "main_THZ1_Nanog_HDAC3_rep3_260806"),
    ("260731_Nanog_p300", "THZ1", "main_THZ1_Nanog_p300_rep1_260731"),
]
manuscript_export_rows.extend(
    export_main_figure_component(experiment_key, comparison, stem)
    for experiment_key, comparison, stem in main_component_specs
)

manuscript_export_manifest = pd.DataFrame(manuscript_export_rows)
manuscript_export_manifest_path = (
    MANUSCRIPT_EXPORT_ROOT / "manuscript_figure_export_manifest.csv"
)
manuscript_export_manifest.to_csv(manuscript_export_manifest_path, index=False)

manuscript_export_paths = [
    Path(path)
    for column in ["png_path", "pdf_path"]
    for path in manuscript_export_manifest[column]
]
missing_manuscript_exports = [
    str(path)
    for path in [*manuscript_export_paths, manuscript_export_manifest_path]
    if not path.exists() or path.stat().st_size == 0
]
if missing_manuscript_exports:
    raise AssertionError(f"Missing manuscript figure exports: {missing_manuscript_exports}")

print(f"Exported {len(manuscript_export_manifest)} manuscript figure components to {MANUSCRIPT_EXPORT_ROOT}")
manuscript_export_manifest

## 0717 Nanog H3K27ac check

This table reports the 0717 Nanog H3K27ac fold changes for total and background-subtracted local and nuclear definitions. It is retained as a focused check of the TSA trajectory and does not replace the primary background-subtracted local analysis.


In [ ]:
diagnosis = summary[
    summary.experiment_key.eq("260717_Nanog_H3K27ac")
    & summary.group.isin(["DMSO", "TSA", "RGFP966"])
]
diagnosis.pivot_table(
    index=["metric", "group"], columns="time_min", values="median_fold_change"
).to_csv(TABLE_ROOT / "qc_0717_TSA_signal_definition_diagnosis.csv")
diagnosis[diagnosis.time_min.eq(60)][["metric", "group", "n", "median_fold_change"]].sort_values(["metric", "group"])


## Output check

The final cell confirms that every expected table, main figure, supplementary figure, and raw audit file was created and is not empty. A missing output stops the notebook with an explicit file list.


In [ ]:
expected_tables = [
    "experiment_inventory.csv", "bleaching_correction_factors.csv",
    "per_cell_log2_fold_change.csv", "signal_trajectories_log2.csv",
    "normality_log2_fold_change.csv", "signal_pairwise_tests.csv",
    "dmso_flatness_validation.csv", "dmso_leave_one_field_out_per_cell.csv",
    "dmso_leave_one_field_out_summary.csv",
    "replicate_figure_manifest.csv", "figure_replicate_selection.csv",
    "qc_counts.csv", "qc_0717_TSA_signal_definition_diagnosis.csv",
]
expected = [TABLE_ROOT / name for name in expected_tables]
expected += [Path(path) for path in figure_manifest.main_png]
expected += [Path(path) for path in figure_manifest.main_pdf]
expected += [Path(path) for path in figure_manifest.supplement_png]
expected += [Path(path) for path in figure_manifest.supplement_pdf]
expected += [Path(path) for path in figure_manifest.raw_audit_png]
expected += [RAW_AUDIT_ROOT / "raw_signal_manifest.csv", RAW_AUDIT_ROOT / "README.md"]
expected += manuscript_export_paths
expected += [manuscript_export_manifest_path]
missing = [str(path) for path in expected if not path.exists() or path.stat().st_size == 0]
if missing:
    raise AssertionError(f"Missing outputs: {missing}")
print(f"Validated {len(expected)} outputs, including {len(figure_manifest)} selected main, {len(figure_manifest)} selected supplementary, and {len(manuscript_export_manifest)} manuscript-export figures; all datasets remain in numerical tables.")


## All-replicate pooled figures

This exploratory view pools the cell-level observations from every available
experiment within each gene, SNAP target, and treatment comparison. Cell-level
signals retain the experiment-specific baseline normalization and matched-DMSO
photobleaching correction computed above. Summary statistics and pairwise tests
are then recalculated after pooling. These plots do not model biological
replicate as a random effect and should therefore be interpreted as pooled-cell
visualizations rather than replicate-level inference.

Only PDF files are written, in a dedicated folder separate from the
replicate-resolved manuscript exports.


In [ ]:
from IPython.display import display


POOLED_REPLICATE_EXPORT_ROOT = RESULT_ROOT / "pooled_all_replicates_pdf"
POOLED_REPLICATE_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)


# Pool the already normalized and bleaching-corrected cell-level observations.
pooled_signal_columns = [
    "gene", "display_snap", "comparison", "metric", "group", "time_min"
]
pooled_primary = corrected[corrected.metric.isin(PRIMARY_METRICS)].copy()
pooled_signal_summary = pooled_primary.groupby(
    pooled_signal_columns, dropna=False
).agg(
    n=("log2_fold_change", "count"),
    median_log2_fc=("log2_fold_change", "median"),
    q25_log2_fc=("log2_fold_change", lambda x: x.quantile(0.25)),
    q75_log2_fc=("log2_fold_change", lambda x: x.quantile(0.75)),
    mean_log2_fc=("log2_fold_change", "mean"),
    sem_log2_fc=(
        "log2_fold_change",
        lambda x: x.std(ddof=1) / math.sqrt(x.count()) if x.count() > 1 else np.nan,
    ),
).reset_index()


# Recalculate the same Mann-Whitney tests and family-wise Holm correction after pooling.
pooled_test_columns = ["gene", "display_snap", "comparison", "metric"]
pooled_signal_test_rows = []
for keys, subset in pooled_primary.groupby(pooled_test_columns, sort=True):
    groups = [group for group in GROUP_ORDER if group in set(subset.group)]
    for time in TIME_MIN[1:]:
        at_time = subset[subset.time_min.eq(time)]
        for group_1, group_2 in combinations(groups, 2):
            values_1 = at_time.loc[
                at_time.group.eq(group_1), "log2_fold_change"
            ].dropna().to_numpy()
            values_2 = at_time.loc[
                at_time.group.eq(group_2), "log2_fold_change"
            ].dropna().to_numpy()
            p_value = (
                mannwhitneyu(values_1, values_2, alternative="two-sided").pvalue
                if len(values_1) and len(values_2)
                else np.nan
            )
            pooled_signal_test_rows.append({
                **dict(zip(pooled_test_columns, keys)),
                "time_min": time,
                "group_1": group_1,
                "group_2": group_2,
                "n_1": len(values_1),
                "n_2": len(values_2),
                "median_1": np.median(values_1) if len(values_1) else np.nan,
                "median_2": np.median(values_2) if len(values_2) else np.nan,
                "median_difference_log2": (
                    np.median(values_2) - np.median(values_1)
                    if len(values_1) and len(values_2)
                    else np.nan
                ),
                "mann_whitney_p": p_value,
            })

pooled_signal_tests = pd.DataFrame(pooled_signal_test_rows)
pooled_signal_tests["holm_p"] = pooled_signal_tests.groupby(
    pooled_test_columns
)["mann_whitney_p"].transform(holm_adjust)


# Pool Active/Inactive classifications across all available experiments.
pooled_active_columns = [
    "gene", "display_snap", "comparison", "group", "time_min"
]
pooled_state = data[data.state.isin(["Active", "Inactive"])].copy()
pooled_active = pooled_state.groupby(pooled_active_columns).state.agg(
    n="count", active_n=lambda x: (x == "Active").sum()
).reset_index()
pooled_active["active_fraction"] = pooled_active.active_n / pooled_active.n
pooled_active[["ci_low", "ci_high"]] = pooled_active.apply(
    lambda row: pd.Series(wilson_interval(row.active_n, row.n)), axis=1
)


pooled_experiment_manifest = pooled_primary[
    ["experiment_key", "gene", "display_snap", "comparison"]
].drop_duplicates().groupby(
    ["gene", "display_snap", "comparison"], sort=True
).experiment_key.agg(
    n_experiments="nunique",
    experiment_keys=lambda x: ";".join(sorted(set(x))),
).reset_index()


def pooled_signal_limits(gene, target, comparison, metric):
    subset = pooled_signal_summary[
        pooled_signal_summary.gene.eq(gene)
        & pooled_signal_summary.display_snap.eq(target)
        & pooled_signal_summary.comparison.eq(comparison)
        & pooled_signal_summary.metric.eq(metric)
    ]
    low = min(-0.25, subset.q25_log2_fc.min())
    high = max(0.25, subset.q75_log2_fc.max())
    span = max(1.0, high - low)
    annotation_rows = 2 if comparison == "HDACi" else 1
    low = math.floor((low - 0.08 * span) * 2) / 2
    high = math.ceil((high + (0.16 + 0.10 * annotation_rows) * span) * 2) / 2
    return low, high


def pooled_branch(gene, target, comparison):
    branch_signal = pooled_signal_summary[
        pooled_signal_summary.gene.eq(gene)
        & pooled_signal_summary.display_snap.eq(target)
        & pooled_signal_summary.comparison.eq(comparison)
    ].copy()
    branch_tests = pooled_signal_tests[
        pooled_signal_tests.gene.eq(gene)
        & pooled_signal_tests.display_snap.eq(target)
        & pooled_signal_tests.comparison.eq(comparison)
    ].copy()
    branch_active = pooled_active[
        pooled_active.gene.eq(gene)
        & pooled_active.display_snap.eq(target)
        & pooled_active.comparison.eq(comparison)
    ].copy()
    if branch_signal.empty:
        raise KeyError(f"No pooled data for {comparison}, {gene}, {target}")
    groups = [group for group in GROUP_ORDER if group in set(branch_signal.group)]
    return branch_signal, branch_tests, branch_active, groups


def plot_pooled_signal_panel(
    ax, subset, tests, groups, gene, comparison, target, metric, title
):
    for group in groups:
        values = subset[subset.group.eq(group)].sort_values("time_min")
        x = values.time_min.to_numpy(dtype=float)
        y = values.median_log2_fc.to_numpy(dtype=float)
        ax.fill_between(
            x,
            values.q25_log2_fc.to_numpy(dtype=float),
            values.q75_log2_fc.to_numpy(dtype=float),
            color=COLORS[group],
            alpha=0.12,
            linewidth=0,
        )
        ax.plot(x, y, color=COLORS[group], marker="o", label=group)
    ax.axhline(0, color="0.70", linewidth=0.6, zorder=0)
    ax.set(
        xticks=TIME_MIN,
        ylim=pooled_signal_limits(gene, target, comparison, metric),
        xlabel="Time (min)",
        ylabel=r"Relative SNAP intensity (log$_2$[$I_t$/$I_0$])",
        title=title,
    )
    style_axis(ax)
    annotate_vs_dmso(
        ax,
        tests,
        [group for group in groups if group != "DMSO"],
        include_t0=False,
    )


def save_pooled_pdf_and_show(fig, stem):
    pdf_path = POOLED_REPLICATE_EXPORT_ROOT / f"{stem}.pdf"
    fig.savefig(pdf_path)
    plt.show()
    plt.close(fig)
    return pdf_path


def export_pooled_local_figure(gene, target, comparison, stem):
    branch_signal, branch_tests, _, groups = pooled_branch(gene, target, comparison)
    fig, ax = plt.subplots(1, 1, figsize=SUPPLEMENT_FIGURE_SIZE_INCHES)
    plot_pooled_signal_panel(
        ax,
        branch_signal[branch_signal.metric.eq("local_bg")],
        branch_tests[branch_tests.metric.eq("local_bg")],
        groups,
        gene,
        comparison,
        target,
        "local_bg",
        rf"$\it{{{gene}}}$ local {target}",
    )
    handles, labels = ax.get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=len(groups),
        handlelength=1.6,
        columnspacing=1.0,
    )
    fig.subplots_adjust(left=0.22, right=0.96, bottom=0.32, top=0.78)
    return save_pooled_pdf_and_show(fig, stem)


def export_pooled_main_figure(gene, target, comparison, stem):
    branch_signal, branch_tests, branch_active, groups = pooled_branch(
        gene, target, comparison
    )
    fig, axes = plt.subplots(1, 2, figsize=MAIN_FIGURE_SIZE_INCHES)
    plot_active_panel(axes[0], branch_active, groups, gene)
    plot_pooled_signal_panel(
        axes[1],
        branch_signal[branch_signal.metric.eq("local_bg")],
        branch_tests[branch_tests.metric.eq("local_bg")],
        groups,
        gene,
        comparison,
        target,
        "local_bg",
        f"{gene} local {target}",
    )
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=len(groups),
        handlelength=1.6,
        columnspacing=1.0,
    )
    fig.subplots_adjust(left=0.105, right=0.98, bottom=0.26, top=0.78, wspace=0.42)
    return save_pooled_pdf_and_show(fig, stem)


pooled_pdf_paths = [
    export_pooled_local_figure(
        "Nanog", "H3K27ac", "HDACi",
        "all_replicates_HDACi_Nanog_H3K27ac_local",
    ),
    export_pooled_local_figure(
        "Sox2", "H3K27ac", "HDACi",
        "all_replicates_HDACi_Sox2_H3K27ac_local",
    ),
    export_pooled_main_figure(
        "Nanog", "H3K27ac", "THZ1",
        "all_replicates_THZ1_Nanog_H3K27ac_active_and_local",
    ),
    export_pooled_main_figure(
        "Sox2", "H3K27ac", "THZ1",
        "all_replicates_THZ1_Sox2_H3K27ac_active_and_local",
    ),
    export_pooled_main_figure(
        "Nanog", "HDAC1", "THZ1",
        "all_replicates_THZ1_Nanog_HDAC1_active_and_local",
    ),
    export_pooled_main_figure(
        "Nanog", "HDAC3", "THZ1",
        "all_replicates_THZ1_Nanog_HDAC3_active_and_local",
    ),
    export_pooled_main_figure(
        "Nanog", "p300", "THZ1",
        "all_replicates_THZ1_Nanog_p300_active_and_local",
    ),
]

missing_pooled_pdfs = [
    str(path) for path in pooled_pdf_paths
    if not path.exists() or path.stat().st_size == 0
]
if missing_pooled_pdfs:
    raise AssertionError(f"Missing pooled-replicate PDFs: {missing_pooled_pdfs}")

pooled_dmso_comparisons = pooled_signal_tests[
    pooled_signal_tests.group_1.eq("DMSO")
    & pooled_signal_tests.metric.eq("local_bg")
].sort_values(
    ["comparison", "gene", "display_snap", "group_2", "time_min"]
).reset_index(drop=True)

print(
    f"Exported {len(pooled_pdf_paths)} pooled-replicate PDFs to "
    f"{POOLED_REPLICATE_EXPORT_ROOT}"
)
display(pooled_experiment_manifest)
display(pooled_dmso_comparisons)


## All-replicate pooled Active, nuclear, and locus-centered figures

For each gene, SNAP target, and treatment comparison, this section displays the
pooled Active fraction together with the pooled nuclear and locus-centered SNAP
signals. It reuses the pooled cell-level summaries and tests calculated in the
preceding section. The nuclear and locus panels show the median and interquartile
range; significance annotations use the pooled Mann-Whitney tests with the same
Holm correction as above.

Only PDF files are written to a dedicated output folder.


In [ ]:
POOLED_ACTIVE_NUCLEAR_LOCUS_ROOT = (
    RESULT_ROOT / "pooled_all_replicates_active_nuclear_locus_pdf"
)
POOLED_ACTIVE_NUCLEAR_LOCUS_ROOT.mkdir(parents=True, exist_ok=True)
POOLED_THREE_PANEL_SIZE_INCHES = (180 / 25.4, 60 / 25.4)


def save_pooled_three_panel_pdf_and_show(fig, stem):
    pdf_path = POOLED_ACTIVE_NUCLEAR_LOCUS_ROOT / f"{stem}.pdf"
    fig.savefig(pdf_path)
    plt.show()
    plt.close(fig)
    return pdf_path


def export_pooled_active_nuclear_locus(gene, target, comparison, stem):
    branch_signal, branch_tests, branch_active, groups = pooled_branch(
        gene, target, comparison
    )
    fig, axes = plt.subplots(1, 3, figsize=POOLED_THREE_PANEL_SIZE_INCHES)

    plot_active_panel(axes[0], branch_active, groups, gene)
    plot_pooled_signal_panel(
        axes[1],
        branch_signal[branch_signal.metric.eq("nuclear_bg")],
        branch_tests[branch_tests.metric.eq("nuclear_bg")],
        groups,
        gene,
        comparison,
        target,
        "nuclear_bg",
        rf"$\it{{{gene}}}$ nuclear {target}",
    )
    plot_pooled_signal_panel(
        axes[2],
        branch_signal[branch_signal.metric.eq("local_bg")],
        branch_tests[branch_tests.metric.eq("local_bg")],
        groups,
        gene,
        comparison,
        target,
        "local_bg",
        rf"$\it{{{gene}}}$ local {target}",
    )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.02),
        ncol=len(groups),
        handlelength=1.6,
        columnspacing=1.0,
    )
    fig.subplots_adjust(
        left=0.07,
        right=0.985,
        bottom=0.26,
        top=0.78,
        wspace=0.52,
    )
    return save_pooled_three_panel_pdf_and_show(fig, stem)


pooled_active_nuclear_locus_pdf_paths = [
    export_pooled_active_nuclear_locus(
        "Nanog", "H3K27ac", "HDACi",
        "all_replicates_HDACi_Nanog_H3K27ac_active_nuclear_locus",
    ),
    export_pooled_active_nuclear_locus(
        "Sox2", "H3K27ac", "HDACi",
        "all_replicates_HDACi_Sox2_H3K27ac_active_nuclear_locus",
    ),
    export_pooled_active_nuclear_locus(
        "Nanog", "H3K27ac", "THZ1",
        "all_replicates_THZ1_Nanog_H3K27ac_active_nuclear_locus",
    ),
    export_pooled_active_nuclear_locus(
        "Sox2", "H3K27ac", "THZ1",
        "all_replicates_THZ1_Sox2_H3K27ac_active_nuclear_locus",
    ),
    export_pooled_active_nuclear_locus(
        "Nanog", "HDAC1", "THZ1",
        "all_replicates_THZ1_Nanog_HDAC1_active_nuclear_locus",
    ),
    export_pooled_active_nuclear_locus(
        "Nanog", "HDAC3", "THZ1",
        "all_replicates_THZ1_Nanog_HDAC3_active_nuclear_locus",
    ),
    export_pooled_active_nuclear_locus(
        "Nanog", "p300", "THZ1",
        "all_replicates_THZ1_Nanog_p300_active_nuclear_locus",
    ),
]

missing_pooled_three_panel_pdfs = [
    str(path)
    for path in pooled_active_nuclear_locus_pdf_paths
    if not path.exists() or path.stat().st_size == 0
]
if missing_pooled_three_panel_pdfs:
    raise AssertionError(
        "Missing pooled Active/nuclear/locus PDFs: "
        f"{missing_pooled_three_panel_pdfs}"
    )

print(
    f"Exported {len(pooled_active_nuclear_locus_pdf_paths)} pooled "
    f"Active/nuclear/locus PDFs to {POOLED_ACTIVE_NUCLEAR_LOCUS_ROOT}"
)
pooled_active_nuclear_locus_pdf_paths
